# Demo 08 — Multi-tenancy (default + partner)

Live check that **two** AITenants are running with **separate** gateways, catalogs, and API keys.

| Tenant | Gateway | Entitlement ns | Models | Persona |
|--------|---------|----------------|--------|---------|
| default | `https://maas.<domain>` | `models-as-a-service` | Granite + llm-katan sims in `llm` | alice / research |
| **partner** | `https://partner-maas.<domain>` | `ai-tenant-partner` | Mistral + Llama in `llm-partner` | bob / apps |

**Prereq:** `./demos/08-maas-35-features/scripts/apply-partner-tenant.sh --with-default` succeeded. Python 3.9+ stdlib. `oc` stays on your laptop.


## Cluster checklist (`oc` on your laptop)

```bash
oc get aitenant -n ai-tenants
oc get maastenantconfig -A | grep -E 'models-as-a-service|ai-tenant-partner'
oc get gateway maas-default-gateway partner -n openshift-ingress
oc get maasmodelref -n llm
oc get maasmodelref -n llm-partner
oc get maassubscription,maasauthpolicy -n models-as-a-service | grep demo08
oc get maassubscription,maasauthpolicy -n ai-tenant-partner
```


## Demo quick swap

Paste both gateway origins and (optionally) pre-minted keys. Leave keys empty to mint in the next cells (needs OpenShift tokens for alice / bob).

| Variable | Meaning |
|----------|---------|
| `DEMO_DEFAULT_BASE` | Default gateway, e.g. `https://maas.apps…` |
| `DEMO_PARTNER_BASE` | Partner gateway, e.g. `https://partner-maas.apps…` |
| `DEMO_DEFAULT_API_KEY` | alice key for `demo08-hybrid-catalog` (optional) |
| `DEMO_PARTNER_API_KEY` | bob key for `demo08-partner-catalog` (optional) |
| `DEMO_ALICE_TOKEN` / `DEMO_BOB_TOKEN` | OpenShift tokens to mint keys if API keys empty |


In [ ]:
DEMO_DEFAULT_BASE = ""
DEMO_PARTNER_BASE = ""
DEMO_DEFAULT_API_KEY = ""
DEMO_PARTNER_API_KEY = ""
DEMO_ALICE_TOKEN = ""
DEMO_BOB_TOKEN = ""


## Setup helpers


In [ ]:
import json
import os
import ssl
import urllib.error
import urllib.request
from typing import Any, Dict, Optional

def _pick(name: str, env: str, default: str = "") -> str:
    v = globals().get(name, "")
    if isinstance(v, str) and v.strip():
        return v.strip().rstrip("/")
    return (os.environ.get(env) or default).strip().rstrip("/")

DEFAULT_BASE = _pick("DEMO_DEFAULT_BASE", "MAAS_BASE", "https://maas.YOUR_DOMAIN_HERE")
PARTNER_BASE = _pick("DEMO_PARTNER_BASE", "MAAS_PARTNER_BASE", "https://partner-maas.YOUR_DOMAIN_HERE")
DEFAULT_API_KEY = _pick("DEMO_DEFAULT_API_KEY", "MAAS_API_KEY") or _pick("DEMO_DEFAULT_API_KEY", "API_KEY")
PARTNER_API_KEY = _pick("DEMO_PARTNER_API_KEY", "MAAS_PARTNER_API_KEY")
ALICE_TOKEN = _pick("DEMO_ALICE_TOKEN", "OPENSHIFT_TOKEN")
BOB_TOKEN = _pick("DEMO_BOB_TOKEN", "OPENSHIFT_BOB_TOKEN")
VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")


def http_json(method: str, url: str, *, token: Optional[str] = None, data: Optional[Dict[str, Any]] = None):
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    body = json.dumps(data).encode("utf-8") if data is not None else None
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            raw = resp.read().decode("utf-8")
            return resp.status, json.loads(raw) if raw else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode("utf-8", errors="replace")
        try:
            parsed = json.loads(err_body) if err_body else {}
        except json.JSONDecodeError:
            parsed = {"_raw": err_body}
        raise RuntimeError(f"HTTP {e.code}: {parsed}") from None


def mint_key(base: str, openshift_token: str, subscription: str, name: str) -> str:
    _, body = http_json(
        "POST",
        f"{base}/maas-api/v1/api-keys",
        token=openshift_token,
        data={
            "name": name,
            "description": "Demo 08 tenancy notebook",
            "expiresIn": "24h",
            "subscription": subscription,
        },
    )
    key = body.get("key")
    if not key:
        raise RuntimeError(f"No key in response: {body}")
    return key


print("DEFAULT_BASE :", DEFAULT_BASE)
print("PARTNER_BASE :", PARTNER_BASE)
print("VERIFY_TLS   :", VERIFY_TLS)


## Mint or reuse API keys


In [ ]:
if not DEFAULT_API_KEY:
    if not ALICE_TOKEN:
        raise SystemExit("Set DEMO_DEFAULT_API_KEY or DEMO_ALICE_TOKEN (alice OpenShift token).")
    DEFAULT_API_KEY = mint_key(DEFAULT_BASE, ALICE_TOKEN, "demo08-hybrid-catalog", "demo08-tenancy-alice")
    print("Minted default-tenant key (prefix):", DEFAULT_API_KEY[:20] + "…")
else:
    print("Using provided default-tenant API key")

if not PARTNER_API_KEY:
    if not BOB_TOKEN:
        raise SystemExit("Set DEMO_PARTNER_API_KEY or DEMO_BOB_TOKEN (bob OpenShift token).")
    PARTNER_API_KEY = mint_key(PARTNER_BASE, BOB_TOKEN, "demo08-partner-catalog", "demo08-tenancy-bob")
    print("Minted partner-tenant key (prefix):", PARTNER_API_KEY[:20] + "…")
else:
    print("Using provided partner-tenant API key")


## Catalogs — each tenant only sees its own models


In [ ]:
_, default_models = http_json("GET", f"{DEFAULT_BASE}/maas-api/v1/models", token=DEFAULT_API_KEY)
_, partner_models = http_json("GET", f"{PARTNER_BASE}/maas-api/v1/models", token=PARTNER_API_KEY)

print("=== Default tenant catalog (alice) ===")
for m in default_models.get("data") or []:
    print(" -", m.get("id") or m.get("name"), "→", m.get("url"))

print()
print("=== Partner tenant catalog (bob) ===")
for m in partner_models.get("data") or []:
    print(" -", m.get("id") or m.get("name"), "→", m.get("url"))


## Isolation — cross-tenant keys must fail


In [ ]:
def status_only(url: str, token: str) -> int:
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(
        url,
        headers={"Authorization": f"Bearer {token}", "Accept": "application/json"},
        method="GET",
    )
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=60) as resp:
            return resp.status
    except urllib.error.HTTPError as e:
        return e.code


cross_1 = status_only(f"{DEFAULT_BASE}/maas-api/v1/models", PARTNER_API_KEY)
cross_2 = status_only(f"{PARTNER_BASE}/maas-api/v1/models", DEFAULT_API_KEY)

print(f"Partner key → default gateway /models : HTTP {cross_1}  (expect 401/403)")
print(f"Default key → partner gateway /models : HTTP {cross_2}  (expect 401/403)")

if cross_1 in (200,) or cross_2 in (200,):
    print("WARNING: cross-tenant access succeeded — check AITenant / maas-api isolation.")
else:
    print("OK: catalogs are isolated across tenants.")


## Partner BBR smoke (optional)


In [ ]:
data = partner_models.get("data") or []
if not data:
    raise SystemExit("Partner catalog empty — check partner-models.yaml and entitlements.")

mid = data[0].get("id") or data[0].get("name")
status, body = http_json(
    "POST",
    f"{PARTNER_BASE}/v1/chat/completions",
    token=PARTNER_API_KEY,
    data={
        "model": mid,
        "messages": [{"role": "user", "content": "Say hello from the partner tenant."}],
        "max_tokens": 32,
    },
)
choices = body.get("choices") or []
msg = (choices[0].get("message") or {}) if choices and isinstance(choices[0], dict) else {}
print("HTTP", status, "| model:", mid)
print("Assistant:", msg.get("content") or "(empty)")
